# Omaha Hi-Lo (8-or-Better) — Text-Based Poker

Full game engine built on PokerKit with configurable **pot-limit** or **fixed-limit** modes.

**Layers:**
- `GameEngine` — PokerKit wrapper; no display/input logic
- `AbstractBot` / `RandomBot` — bot base class and random-action bot
- `HumanPlayer` — prompts via `input()` (swap for WebSocket later)
- `TableDisplay` — all rendering
- `GameController` — orchestrates hands, rotation, bust-outs

**Cell 8** runs 2,000 hands with two RandomBots and saves a JSON hand-history log.

In [1]:
# ============================================================
# CELL 1 — CONFIGURATION  (edit here before running)
# ============================================================
CONFIG = {
    # Betting structure
    'game_type': 'pot_limit',      # 'pot_limit' | 'fixed_limit'
    'fixed_small_bet': 2,          # fixed-limit pre-flop / flop bet size
    'fixed_big_bet':   4,          # fixed-limit turn / river bet size

    # Table
    'num_seats':  2,               # 2–6 total seats
    'num_humans': 1,               # 0–2
    'num_bots':   1,               # 1–4

    # Stacks & blinds
    'starting_stack': 200,
    'small_blind': 1,
    'big_blind':   2,

    # Seat-rotation prompt interval (every N hands)
    'seat_rotation_interval': 10,
}

assert CONFIG['num_humans'] + CONFIG['num_bots'] == CONFIG['num_seats'], \
    'num_humans + num_bots must equal num_seats'
assert 2 <= CONFIG['num_seats'] <= 6, 'num_seats must be 2–6'
print('Config OK:', CONFIG)

Config OK: {'game_type': 'pot_limit', 'fixed_small_bet': 2, 'fixed_big_bet': 4, 'num_seats': 2, 'num_humans': 1, 'num_bots': 1, 'starting_stack': 200, 'small_blind': 1, 'big_blind': 2, 'seat_rotation_interval': 10}


In [2]:
# ============================================================
# CELL 2 — IMPORTS + POT-LIMIT HI-LO GAME CLASS
# ============================================================
from __future__ import annotations

import json
import random
import time
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from typing import Optional

from pokerkit import (
    Automation,
    FixedLimitOmahaHoldemHighLowSplitEightOrBetter,
    OmahaEightOrBetterLowHand,
    OmahaHoldemHand,
    PotLimitOmahaHoldem,
    State,
)


class PotLimitOmahaHoldemHighLowSplitEightOrBetter(PotLimitOmahaHoldem):
    """Pot-limit Omaha Hi-Lo 8-or-better.

    Inherits all pot-limit betting mechanics from PotLimitOmahaHoldem but
    evaluates both the best Omaha high hand *and* the best qualifying low
    hand (8-or-better) at showdown, splitting pots accordingly.
    Side-pots and all-in equity are handled internally by PokerKit.
    """
    hand_types = (OmahaHoldemHand, OmahaEightOrBetterLowHand)


print('PokerKit game classes ready.')

PokerKit game classes ready.


In [3]:
# ============================================================
# CELL 3 — ACTION DATA CLASS + GAME ENGINE
# ============================================================

# ── Action ──────────────────────────────────────────────────

@dataclass
class Action:
    """A single player action.

    action_type: 'fold' | 'check' | 'call' | 'raise'
    amount:      total bet-to amount for raises; None means min-raise.
    """
    action_type: str
    amount: Optional[int] = None


# ── GameEngine ───────────────────────────────────────────────

class GameEngine:
    """Wraps PokerKit for Omaha Hi-Lo 8-or-better.

    Manages the deck, board, pots (including side pots for all-in players),
    blinds, dealing, and showdown logic (hi/lo split vs. scoop).
    PokerKit enforces the Omaha rule of exactly 2 hole + 3 board cards.

    This class knows nothing about display or input.

    Typical usage::

        engine = GameEngine(CONFIG)
        gs = engine.start_hand(player_names, stacks)
        while not gs['hand_over']:
            action = ...          # decided externally
            gs = engine.submit_action(gs['actor_index'], action)
    """

    _AUTOMATIONS = (
        Automation.ANTE_POSTING,
        Automation.BET_COLLECTION,
        Automation.BLIND_OR_STRADDLE_POSTING,
        Automation.HOLE_DEALING,
        Automation.BOARD_DEALING,
        Automation.CARD_BURNING,
        Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
        Automation.HAND_KILLING,
        Automation.CHIPS_PUSHING,
        Automation.CHIPS_PULLING,
    )

    _STREET = {0: 'preflop', 3: 'flop', 4: 'turn', 5: 'river'}

    def __init__(self, config: dict) -> None:
        self._cfg = config
        self._state: Optional[State] = None
        self._saved_holes: list[list[str]] = []
        self._player_names: list[str] = []

    # ── public API ───────────────────────────────────────────

    def start_hand(self, player_names: list[str], stacks: list[int]) -> dict:
        """Initialise a new hand and return the opening game state.

        player_names[0] sits in the SB seat (BTN/SB in heads-up).
        Blinds are posted and all 4 hole cards are dealt automatically.
        """
        n = len(player_names)
        self._player_names = list(player_names)
        sb, bb = self._cfg['small_blind'], self._cfg['big_blind']

        # PokerKit reverses blind assignments for 2-player heads-up, so
        # we pass (bb, sb) to achieve pos-0 = SB (1 chip), pos-1 = BB (2 chips).
        # For 3+ players the tuple order matches position order directly.
        if n == 2:
            blinds: tuple = (bb, sb)
        else:
            blinds = tuple(sb if i == 0 else bb if i == 1 else 0 for i in range(n))

        if self._cfg['game_type'] == 'fixed_limit':
            self._state = FixedLimitOmahaHoldemHighLowSplitEightOrBetter.create_state(
                self._AUTOMATIONS, True, 0, blinds,
                self._cfg['fixed_small_bet'], self._cfg['fixed_big_bet'],
                tuple(stacks), n,
            )
        else:
            self._state = PotLimitOmahaHoldemHighLowSplitEightOrBetter.create_state(
                self._AUTOMATIONS, True, 0, blinds,
                bb, tuple(stacks), n,
            )

        # Save hole cards now; they may be cleared at showdown by PokerKit.
        self._saved_holes = [
            [repr(c) for c in hand] for hand in self._state.hole_cards
        ]
        return self.get_game_state()

    def get_game_state(self) -> dict:
        """Return a plain-dict snapshot of the current public game state.

        Keys:
            stacks        – chip counts per PokerKit position
            bets          – chips already committed this street
            pot           – total chips in all pots
            board         – community cards as strings, e.g. ['As','2h','3c']
            hole_cards    – list of 4-card lists; always populated (saved at deal)
            actor_index   – whose turn it is (None if hand is over)
            legal_actions – dict of legal moves and amounts (empty if no actor)
            street        – 'preflop' | 'flop' | 'turn' | 'river'
            player_names  – names in PokerKit position order
            sb_pos        – PokerKit index of SB (always 0)
            bb_pos        – PokerKit index of BB (always 1)
            dealer_pos    – PokerKit index of BTN (0 heads-up, n-1 for 3+)
            hand_over     – True once all betting and payouts are complete
            payoffs       – net chip changes per position (only when hand_over)
            player_count  – number of seats this hand
        """
        st = self._state
        assert st is not None, 'Call start_hand() first'

        board = [repr(c) for grp in st.board_cards for c in grp]
        n = len(self._player_names)
        actor = st.actor_index
        hand_over = not st.status

        legal: dict = {}
        if actor is not None and not hand_over:
            call_amt = st.checking_or_calling_amount or 0
            can_coc  = st.can_check_or_call()
            can_raise = st.can_complete_bet_or_raise_to()
            legal = {
                'can_fold':   st.can_fold(),
                'can_check':  can_coc and call_amt == 0,
                'can_call':   can_coc and call_amt > 0,
                'call_amount': call_amt,
                'can_raise':  can_raise,
                'min_raise':  st.min_completion_betting_or_raising_to_amount,
                'max_raise':  st.max_completion_betting_or_raising_to_amount,
                # pot-sized raise (same as max for pot-limit; may differ fixed-limit)
                'pot_raise':  st.pot_completion_betting_or_raising_to_amount,
            }

        return {
            'stacks':       list(st.stacks),
            'bets':         list(st.bets),
            'pot':          st.total_pot_amount,
            'board':        board,
            'hole_cards':   self._saved_holes,
            'actor_index':  actor,
            'legal_actions': legal,
            'street':       self._STREET.get(len(board), f'street_{len(board)}'),
            'player_names': list(self._player_names),
            'sb_pos':       0,
            'bb_pos':       1,
            'dealer_pos':   0 if n == 2 else n - 1,
            'hand_over':    hand_over,
            'payoffs':      list(st.payoffs) if hand_over else None,
            'player_count': n,
        }

    def submit_action(self, actor_pos: int, action: Action) -> dict:
        """Apply action from the player at actor_pos and return the new state.

        Raises AssertionError if it is not actor_pos's turn.
        For raises, action.amount is the total bet-to amount; None means min-raise.
        """
        st = self._state
        assert st is not None, 'Call start_hand() first'
        assert st.actor_index == actor_pos, (
            f'Expected actor {st.actor_index}, got {actor_pos}'
        )

        t = action.action_type
        if t == 'fold':
            st.fold()
        elif t in ('check', 'call'):
            st.check_or_call()
        elif t == 'raise':
            amt = action.amount
            if amt is None:
                amt = st.min_completion_betting_or_raising_to_amount
            st.complete_bet_or_raise_to(amt)
        else:
            raise ValueError(f'Unknown action type: {t!r}')

        return self.get_game_state()


print('GameEngine defined.')

GameEngine defined.


In [4]:
# ============================================================
# CELL 4 — ABSTRACT BOT + RANDOM BOT + HUMAN PLAYER
# ============================================================

class AbstractBot(ABC):
    """Base class for all player agents.

    Subclass and implement decide().  New bot types need only override that
    one method; the constructor signature is intentionally fixed so the
    GameController can create bots uniformly.
    """

    def __init__(self, name: str, starting_stack: int, bot_type: str = 'abstract') -> None:
        self.name           = name
        self.starting_stack = starting_stack
        self.bot_type       = bot_type

    @abstractmethod
    def decide(self, game_state: dict, my_pos: int) -> Action:
        """Return an Action for this player's turn.

        game_state:  full snapshot from GameEngine.get_game_state()
        my_pos:      this player's index within game_state arrays
                     (game_state['hole_cards'][my_pos] are this player's cards)
        """
        ...


# ── RandomBot ───────────────────────────────────────────────

_RANDOM_BOT_MAX_RAISE = 400  # chips; caps pot-limit raise size to control variance


class RandomBot(AbstractBot):
    """Selects uniformly at random from all legal actions.

    When raising, picks a random legal amount up to min(max_raise, 400).
    The 400-chip cap keeps pots bounded and prevents instant bust-outs in
    pot-limit mode during long simulation runs.
    """

    def __init__(self, name: str, starting_stack: int) -> None:
        super().__init__(name, starting_stack, bot_type='random')

    def decide(self, game_state: dict, my_pos: int) -> Action:
        legal = game_state['legal_actions']
        pool: list[str] = []
        if legal.get('can_fold'):  pool.append('fold')
        if legal.get('can_check'): pool.append('check')
        if legal.get('can_call'):  pool.append('call')
        if legal.get('can_raise'): pool.append('raise')
        if not pool:
            pool = ['check']  # fallback: should not happen

        choice = random.choice(pool)
        if choice == 'raise':
            lo = int(legal['min_raise'])
            hi = min(int(legal['max_raise']), _RANDOM_BOT_MAX_RAISE)
            hi = max(hi, lo)  # ensure hi >= lo even when cap is below min_raise
            amt = random.randint(lo, hi) if lo != hi else lo
            return Action('raise', amt)
        return Action(choice)


# ── HumanPlayer ─────────────────────────────────────────────

class HumanPlayer(AbstractBot):
    """Prompts for input via input().  Replace decide() for WebSocket use."""

    def __init__(self, name: str, starting_stack: int) -> None:
        super().__init__(name, starting_stack, bot_type='human')

    def decide(self, game_state: dict, my_pos: int) -> Action:
        legal    = game_state['legal_actions']
        my_cards = game_state['hole_cards'][my_pos]

        from omaha_player_helpers import _fmt_cards  # defined in Cell 5
        print(f'\n  Your hole cards: {_fmt_cards(my_cards)}')

        # Build option strings for the prompt
        options: list[str] = []
        if legal.get('can_fold'):  options.append('fold')
        if legal.get('can_check'): options.append('check')
        if legal.get('can_call'):  options.append(f"call {legal['call_amount']}")
        if legal.get('can_raise'):
            lo, hi = legal['min_raise'], legal['max_raise']
            options.append(f'raise {lo}–{hi}')

        prompt = '  Action [' + ' / '.join(options) + ']: '

        while True:
            raw   = input(prompt).strip().lower()
            parts = raw.split()
            if not parts:
                continue
            cmd = parts[0]

            if cmd == 'fold' and legal.get('can_fold'):
                return Action('fold')
            elif cmd in ('check', 'x') and legal.get('can_check'):
                return Action('check')
            elif cmd in ('call', 'c') and legal.get('can_call'):
                return Action('call')
            elif cmd in ('raise', 'r', 'bet', 'b') and legal.get('can_raise'):
                lo, hi = legal['min_raise'], legal['max_raise']
                if len(parts) >= 2:
                    try:
                        amt = int(parts[1])
                    except ValueError:
                        print(f'  Enter a number between {lo} and {hi}.')
                        continue
                else:
                    # Default to pot-raise (pot-limit) or min-raise
                    amt = legal.get('pot_raise') or lo
                if int(lo) <= amt <= int(hi):
                    return Action('raise', amt)
                print(f'  Amount must be between {lo} and {hi}.')
            else:
                print(f"  Unrecognised. Options: {', '.join(options)}")


print('AbstractBot / RandomBot / HumanPlayer defined.')


AbstractBot / RandomBot / HumanPlayer defined.


In [5]:
# ============================================================
# CELL 5 — TABLE DISPLAY
# ============================================================

# Suit symbols for nicer rendering
_SUIT_SYM = {'s': '\u2660', 'h': '\u2665', 'd': '\u2666', 'c': '\u2663'}


def _fmt_card(c: str) -> str:
    """Turn '9s' into '9\u2660', 'Ah' into 'A\u2665', etc."""
    if len(c) < 2:
        return c
    return c[:-1] + _SUIT_SYM.get(c[-1].lower(), c[-1])


def _fmt_cards(cards: list[str]) -> str:
    return '  '.join(_fmt_card(c) for c in cards)


# Monkey-patch the helper import used in HumanPlayer.decide
import sys, types
_m = types.ModuleType('omaha_player_helpers')
_m._fmt_cards = _fmt_cards
sys.modules['omaha_player_helpers'] = _m


class TableDisplay:
    """Renders the text UI.  Knows about display only — no game logic.

    human_positions: set of PokerKit position indices that are HumanPlayer.
    Bot hole cards are shown as '[hidden]' during play and revealed at showdown.
    """

    _SEP  = '─' * 62
    _SEP2 = '═' * 62

    # ── position labels ──────────────────────────────────────

    @staticmethod
    def _pos_label(pos: int, gs: dict) -> str:
        n = gs['player_count']
        tags: list[str] = []
        if pos == gs['dealer_pos']:
            tags.append('BTN' if n > 2 else 'BTN/SB')
        if pos == gs['sb_pos'] and n > 2:
            tags.append('SB')
        if pos == gs['bb_pos']:
            tags.append('BB')
        return ' '.join(tags)

    # ── render helpers ───────────────────────────────────────

    def _player_row(self, pos: int, gs: dict,
                    human_positions: set[int],
                    reveal_all: bool = False) -> str:
        name  = gs['player_names'][pos]
        stack = gs['stacks'][pos]
        bet   = gs['bets'][pos]
        label = self._pos_label(pos, gs)
        is_actor = (gs['actor_index'] == pos)
        arrow = '→' if is_actor else ' '

        if reveal_all or pos in human_positions:
            cards_str = _fmt_cards(gs['hole_cards'][pos])
        else:
            cards_str = '[hidden]'

        bet_str = f'  bet={bet}' if bet else ''
        return (f'  {arrow}[{pos}] {name:<14}  stack={stack:>5}{bet_str}'
                f'  {cards_str}  {label}')

    # ── public render methods ────────────────────────────────

    def render_new_hand(self, gs: dict, hand_num: int,
                        human_positions: set[int]) -> None:
        """Print the start-of-hand header with all seats."""
        print(f'\n{self._SEP2}')
        print(f'  Hand #{hand_num}   street={gs["street"].upper()}   '
              f'pot={gs["pot"]}   mode={CONFIG["game_type"].replace("_","-")}')
        print(self._SEP)
        for pos in range(gs['player_count']):
            print(self._player_row(pos, gs, human_positions))
        print(self._SEP)

    def render_street_change(self, gs: dict,
                             human_positions: set[int]) -> None:
        """Print the new community cards when a street begins."""
        board_str = _fmt_cards(gs['board']) if gs['board'] else '(none)'
        print(f'\n  ── {gs["street"].upper()} ──  '
              f'Board: {board_str}   pot={gs["pot"]}')
        for pos in range(gs['player_count']):
            stack, bet = gs['stacks'][pos], gs['bets'][pos]
            bet_str = f'  bet={bet}' if bet else ''
            print(f'  [{pos}] {gs["player_names"][pos]:<14}  '
                  f'stack={stack:>5}{bet_str}')

    def render_action(self, gs: dict, actor_pos: int,
                      action: Action) -> None:
        """Print a single player action on its own line."""
        name = gs['player_names'][actor_pos]
        if action.action_type == 'fold':
            desc = 'folds'
        elif action.action_type == 'check':
            desc = 'checks'
        elif action.action_type == 'call':
            desc = f"calls {gs['legal_actions']['call_amount']}"
        elif action.action_type == 'raise':
            desc = f'raises to {action.amount}'
        else:
            desc = action.action_type
        print(f'    [{actor_pos}] {name} {desc}   (pot → {gs["pot"]})')

    def render_showdown(self, gs: dict) -> None:
        """Print the final board, all hole cards, and per-player payoffs."""
        board_str = _fmt_cards(gs['board']) if gs['board'] else '(board missing)'
        print(f'\n{self._SEP}')
        print(f'  SHOWDOWN   Board: {board_str}')
        print(self._SEP)
        payoffs = gs['payoffs'] or [0] * gs['player_count']
        for pos in range(gs['player_count']):
            name  = gs['player_names'][pos]
            cards = _fmt_cards(gs['hole_cards'][pos])
            p     = payoffs[pos]
            sign  = '+' if p >= 0 else ''
            print(f'  [{pos}] {name:<14}  {cards:<28}  {sign}{p}')
        print()

    def render_bust(self, name: str) -> None:
        print(f'\n  *** {name} has been ELIMINATED (stack = 0) ***')


print('TableDisplay defined.')

TableDisplay defined.


In [6]:
# ============================================================
# CELL 6 — GAME CONTROLLER
# ============================================================

class GameController:
    """Orchestrates hands: dealer rotation, bust-outs, seat rotation.

    Connects GameEngine (game logic), player agents, and TableDisplay.
    The GameEngine can later be wrapped in a web server without changes.

    Player order inside each hand
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    Seats are numbered 0 … N-1 as stable identifiers.  For each hand
    a *rotation* maps PokerKit positions [0=SB, 1=BB, …, n-1=BTN] to
    absolute seat indices, determined by dealer_seat.
    """

    def __init__(self, config: dict,
                 players: list[AbstractBot]) -> None:
        self.config      = config
        self.players     = list(players)
        self.stacks      = [p.starting_stack for p in players]
        self.dealer_seat = 0    # absolute seat index of current BTN
        self.engine      = GameEngine(config)
        self.display     = TableDisplay()
        self.hand_count  = 0
        self.hand_log: list[dict] = []

    # ── seat management ──────────────────────────────────────

    def _active_seats(self) -> list[int]:
        """Absolute seat indices of players still in the game."""
        return [i for i, s in enumerate(self.stacks) if s > 0]

    def _build_rotation(self) -> list[int]:
        """PokerKit-position → absolute-seat mapping for this hand.

        Returns a list where index i is the absolute seat of PokerKit
        position i (0=SB, 1=BB, …, n-1=BTN).
        """
        active = self._active_seats()
        n = len(active)
        btn = (active.index(self.dealer_seat)
               if self.dealer_seat in active else 0)
        # SB is one seat clockwise from BTN
        return [active[(btn + 1 + i) % n] for i in range(n)]

    def _advance_dealer(self) -> None:
        active = self._active_seats()
        if not active:
            return
        cur = active.index(self.dealer_seat) if self.dealer_seat in active else 0
        self.dealer_seat = active[(cur + 1) % len(active)]

    # ── single hand ──────────────────────────────────────────

    def run_one_hand(self, silent: bool = False) -> dict:
        """Play one complete hand and return a log-entry dict."""
        rotation    = self._build_rotation()
        names       = [self.players[s].name  for s in rotation]
        hand_stacks = [self.stacks[s]         for s in rotation]
        human_pos   = {i for i, s in enumerate(rotation)
                       if isinstance(self.players[s], HumanPlayer)}

        gs = self.engine.start_hand(names, hand_stacks)

        if not silent:
            self.display.render_new_hand(gs, self.hand_count + 1, human_pos)

        prev_street = gs['street']

        while not gs['hand_over']:
            actor_pos = gs['actor_index']
            if actor_pos is None:
                break

            # Announce new street (flop / turn / river)
            if gs['street'] != prev_street:
                if not silent:
                    self.display.render_street_change(gs, human_pos)
                prev_street = gs['street']

            abs_seat = rotation[actor_pos]
            player   = self.players[abs_seat]
            action   = player.decide(gs, actor_pos)

            if not silent:
                self.display.render_action(gs, actor_pos, action)

            gs = self.engine.submit_action(actor_pos, action)

        if not silent:
            self.display.render_showdown(gs)

        # Update global stacks from payoffs
        payoffs  = gs['payoffs'] or [0] * len(rotation)
        pot_size = sum(p for p in payoffs if p > 0)
        for pk_pos, abs_seat in enumerate(rotation):
            self.stacks[abs_seat] += payoffs[pk_pos]

        self.hand_count += 1
        self._advance_dealer()

        entry = {
            'hand':         self.hand_count,
            'board':        gs['board'],
            'hole_cards':   {names[i]: gs['hole_cards'][i]
                             for i in range(len(names))},
            'payoffs':      {names[i]: payoffs[i]
                             for i in range(len(names))},
            'pot':          pot_size,
            'final_stacks': {names[i]: self.stacks[rotation[i]]
                             for i in range(len(names))},
        }
        self.hand_log.append(entry)
        return entry

    # ── bust-out handling ────────────────────────────────────

    def _bust_check(self) -> None:
        """Announce any players whose stack just reached zero."""
        for player, stack in zip(self.players, self.stacks):
            if stack <= 0 and not getattr(player, '_busted', False):
                self.display.render_bust(player.name)
                player._busted = True  # only announce once

    # ── session loop ─────────────────────────────────────────

    def run_session(self, max_hands: Optional[int] = None,
                    silent: bool = False) -> None:
        """Run hands until one player remains or max_hands is reached.

        Every seat_rotation_interval hands, prompt human players to
        shuffle seats (skipped if silent=True).
        """
        while True:
            active = self._active_seats()

            if len(active) < 2:
                if not silent:
                    winner = (self.players[active[0]].name
                              if active else 'Nobody')
                    chips  = self.stacks[active[0]] if active else 0
                    print(f'\n{TableDisplay._SEP2}')
                    print(f'  Game over after {self.hand_count} hands!')
                    print(f'  Winner: {winner}  ({chips} chips)')
                    print(TableDisplay._SEP2)
                break

            if max_hands is not None and self.hand_count >= max_hands:
                break

            self.run_one_hand(silent=silent)
            self._bust_check()

            # Seat-rotation prompt
            ri = self.config['seat_rotation_interval']
            has_human = any(isinstance(p, HumanPlayer)
                            for p in self.players)
            if (not silent and has_human
                    and self.hand_count % ri == 0):
                ans = input(
                    f'\n  [{self.hand_count} hands] Rotate seats? (y/n): '
                ).strip().lower()
                if ans == 'y':
                    active_now  = self._active_seats()
                    pairs       = [(self.players[i], self.stacks[i])
                                   for i in active_now]
                    random.shuffle(pairs)
                    for seat, (p, s) in zip(active_now, pairs):
                        self.players[seat] = p
                        self.stacks[seat]  = s
                    self.dealer_seat = active_now[0]
                    print('  Seats shuffled!')


print('GameController defined.')

GameController defined.


In [ ]:
# ============================================================
# CELL 7 — PLAY  (1 HumanPlayer + 1 RandomBot)
# ============================================================
# Re-read CONFIG from Cell 1 — change game_type / blinds there.

random.seed()   # fresh seed each run

human = HumanPlayer('You',       CONFIG['starting_stack'])
bot   = RandomBot('RandomBot',   CONFIG['starting_stack'])

ctrl = GameController(CONFIG, [human, bot])
ctrl.run_session()


══════════════════════════════════════════════════════════════
  Hand #1   street=PREFLOP   pot=3   mode=pot-limit
──────────────────────────────────────────────────────────────
  →[0] RandomBot       stack=  199  bet=1  [hidden]  BTN/SB
   [1] You             stack=  198  bet=2  8♣  7♥  9♥  J♠  BB
──────────────────────────────────────────────────────────────
    [0] RandomBot raises to 6   (pot → 3)

  Your hole cards: 8♣  7♥  9♥  J♠
    [1] You calls 4   (pot → 8)

  ── FLOP ──  Board: 5♠  K♥  T♣   pot=12
  [0] RandomBot       stack=  194
  [1] You             stack=  194
    [0] RandomBot checks   (pot → 12)

  Your hole cards: 8♣  7♥  9♥  J♠
    [1] You raises to 12   (pot → 12)
    [0] RandomBot raises to 45   (pot → 24)

  Your hole cards: 8♣  7♥  9♥  J♠
    [1] You calls 33   (pot → 69)

  ── TURN ──  Board: 5♠  K♥  T♣  3♣   pot=102
  [0] RandomBot       stack=  149
  [1] You             stack=  149
    [0] RandomBot checks   (pot → 102)

  Your hole cards: 8♣  7♥  9♥  J♠
    

In [7]:
# ============================================================
# CELL 8 — TEST MODE: 2,000 hands, 2 RandomBots, no human
# ============================================================
# Runs silently, prints summary stats, saves hand history JSON.
# If a bot busts before 2,000 hands, a fresh game is started and
# hands accumulate across sessions until the target is reached.

TEST_CONFIG = {
    **CONFIG,
    'num_seats':  2,
    'num_humans': 0,
    'num_bots':   2,
    'starting_stack': 200_000_000,
}
NUM_HANDS = 2_000

random.seed(42)
target     = NUM_HANDS
total_hands = 0
all_log: list[dict] = []
sessions   = 0

t0 = time.perf_counter()
while total_hands < target:
    remaining = target - total_hands
    bot_a = RandomBot('BotA', TEST_CONFIG['starting_stack'])
    bot_b = RandomBot('BotB', TEST_CONFIG['starting_stack'])
    tc = GameController(TEST_CONFIG, [bot_a, bot_b])
    tc.run_session(max_hands=remaining, silent=True)
    total_hands += tc.hand_count
    all_log.extend(tc.hand_log)
    sessions += 1
elapsed = time.perf_counter() - t0

# ── Summary stats ────────────────────────────────────────────
wins:      dict[str, int] = defaultdict(int)
pot_sizes: list[int]      = []

for entry in all_log:
    best = max(entry['payoffs'], key=lambda k: entry['payoffs'][k])
    if entry['payoffs'][best] > 0:
        wins[best] += 1
    pot_sizes.append(entry['pot'])

sep = '=' * 52
print(f'\n{sep}')
print(f'  TEST RESULTS — {total_hands} hands in {elapsed:.2f}s'
      f'  ({total_hands / elapsed:.0f} h/s)  [{sessions} session(s)]')
print(sep)
avg_pot = sum(pot_sizes) / len(pot_sizes) if pot_sizes else 0
print(f'  Avg pot size : {avg_pot:.2f}')
print(f'  BotA wins    : {wins["BotA"]}')
print(f'  BotB wins    : {wins["BotB"]}')
print(sep)

# ── Save hand history as JSON (for ML training / analysis) ───
HISTORY_FILE = 'omaha_hand_history.json'
with open(HISTORY_FILE, 'w') as f:
    json.dump(all_log, f, indent=2)
print(f'\n  Hand history ({len(all_log)} entries) saved → {HISTORY_FILE}')


NameError: name 'GameEngine' is not defined